In [2]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

koleksi_dokumen = {
    1: "Pengembangan Sistem Absensi QR Code Berbasis Web untuk Siswa Jurusan Teknik Komputer dan Jaringan SMK Negeri 4 Makassar",
    2: "Analisis Dampak Penggunaan ChatGPT terhadap Pencapaian Akademik Siswa Jurusan Teknik Komputer dan Jaringan di SMK Negeri 2 Makassar",
    3: "Pengembangan E-Modul Internet of Things Program Studi Teknik Komputer Fakultas Teknik Universitas Negeri Makassar",
    4: "Pengembangan Sistem Kontrol Lampu dan Pengamanan Pintu Rumah dengan Monitoring Kamera Berbasis ESP32 Terintegrasi Aplikasi Android",
    5: "Pengaruh ChatGPT Dependency terhadap Kemampuan Literasi Digital Mahasiswa Jurusan Teknik Informatika dan Komputer",
    6: "Pengembangan E-Modul Interaktif Berbasis Flipbook pada Mata Kuliah Elektronika Analog dan Digital Jurusan Teknik Informatika dan Komputer",
    7: "Analisis Kerentanan Website E-Skripsi JTIK dan Rancangan Solusi Keamanan Berdasarkan OWASP WSTG",
    8: "Pengembangan Virtual Assistant Chatbot Berbasis WhatsApp untuk Pembelajaran Melalui Teknologi ChatGPT dengan Integrasi Replit dan Uptime Robot di SMK Negeri 01 Barru",
    9: "Perancangan E-Modul Mata Kuliah Teknologi Virtualisasi dan Cloud pada Program Studi Teknik Komputer Universitas Negeri Makassar",
    10: "Pengaruh Game Online Mobile Legends terhadap Prestasi Belajar Mahasiswa Jurusan Teknik Informatika dan Komputer"
}

ground_truth = {
    "AI": {2, 5, 8},
    "Jaringan": {1, 2},
    "IoT": {3, 4}
}

dokumen_list = list(koleksi_dokumen.values())
doc_ids = list(koleksi_dokumen.keys())

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(dokumen_list)

def hitung_metrik(query, gt_set, k=3):
    query_vec = vectorizer.transform([query])
    skor_sim = cosine_similarity(query_vec, X).flatten()

    sorted_indices = np.argsort(skor_sim)[::-1]
    hasil_sistem = [doc_ids[idx] for idx in sorted_indices if skor_sim[idx] > 0]

    top_k = hasil_sistem[:k]
    p_at_k = len(set(top_k) & gt_set) / k

    total_rel = len(gt_set)
    rel_retrieved = len(set(hasil_sistem) & gt_set)
    recall = rel_retrieved / total_rel if total_rel > 0 else 0.0

    f1 = (2 * p_at_k * recall) / (p_at_k + recall) if (p_at_k + recall) > 0 else 0.0

    hits = 0
    sum_precisions = 0.0
    for i, doc_id in enumerate(hasil_sistem, start=1):
        if doc_id in gt_set:
            hits += 1
            sum_precisions += hits / i
    ap = sum_precisions / total_rel if total_rel > 0 else 0.0

    return p_at_k, recall, f1, ap, hasil_sistem

ap_list = []
print("=== HASIL EVALUASI SISTEM TEMU KEMBALI ===")
for q, gt in ground_truth.items():
    p_k, rec, f1, ap, ret = hitung_metrik(q, gt, k=3)
    ap_list.append(ap)
    print(f"Query: '{q}'")
    print(f"  - Dokumen Dikembalikan : {ret}")
    print(f"  - Precision@3          : {p_k:.4f}")
    print(f"  - Recall               : {rec:.4f}")
    print(f"  - F1-Score             : {f1:.4f}")
    print(f"  - Average Precision    : {ap:.4f}\n")

map_score = sum(ap_list) / len(ap_list)
print(f"Mean Average Precision (MAP): {map_score:.4f}")

=== HASIL EVALUASI SISTEM TEMU KEMBALI ===
Query: 'AI'
  - Dokumen Dikembalikan : []
  - Precision@3          : 0.0000
  - Recall               : 0.0000
  - F1-Score             : 0.0000
  - Average Precision    : 0.0000

Query: 'Jaringan'
  - Dokumen Dikembalikan : [1, 2]
  - Precision@3          : 0.6667
  - Recall               : 1.0000
  - F1-Score             : 0.8000
  - Average Precision    : 1.0000

Query: 'IoT'
  - Dokumen Dikembalikan : []
  - Precision@3          : 0.0000
  - Recall               : 0.0000
  - F1-Score             : 0.0000
  - Average Precision    : 0.0000

Mean Average Precision (MAP): 0.3333
